# Bronze Ingestion - Sentinel-1 (Microsoft Planetary Computer)

**Workload:** Geohazard demo - Bronze layer  
**Source:** [Microsoft Planetary Computer STAC API](https://planetarycomputer.microsoft.com/api/stac/v1)  
**Area of Interest (AOI):** Maple Ridge, British Columbia (south coast)

## What this notebook does
Grabs **Sentinel-1 C-band radar (SAR)** scene *metadata* from the Planetary
Computer STAC API for the shared AOI and writes it into bronze Delta tables.
It reuses the same `ingest_collection(...)` helper pattern as
`bronze_pc_collections.ipynb`, focused on the two Sentinel-1 product types.

Sentinel-1 radar sees through cloud and darkness, which makes it valuable for
geohazard monitoring (surface change, moisture, deformation context) where
optical Sentinel-2 is often obscured.

> **Metadata bronze pattern:** we capture the STAC item records (ids,
> footprints, asset lists, polarizations, orbit properties). Downloading the
> underlying COGs is a later silver/gold concern and is intentionally out of
> scope.

## Collections ingested
| STAC collection | Bronze table | Product | Time filter |
| --- | --- | --- | --- |
| `sentinel-1-rtc` | `bronze_sentinel_1_rtc` | Radiometrically Terrain Corrected backscatter | 2024 summer |
| `sentinel-1-grd` | `bronze_sentinel_1_grd` | Ground Range Detected amplitude | 2024 summer |

> `sentinel-1-rtc` is also produced by `bronze_pc_collections.ipynb`; this
> notebook refreshes it with the identical schema (overwrite) and adds the raw
> `sentinel-1-grd` product alongside it.

## 1. Shared configuration and helper
Run this cell once. It defines the AOI, the bounding box, a stable bronze
schema, and the `ingest_collection()` function used by the collection cells
below. Same pattern as `bronze_pc_collections.ipynb`.

In [ ]:
import math
import json
import requests
from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

# --- Area of Interest: Maple Ridge, BC -------------------------------------
LATITUDE = 49.2193
LONGITUDE = -122.5984
RADIUS_KM = 20
AOI_NAME = "Maple Ridge, BC"

STAC_SEARCH_URL = "https://planetarycomputer.microsoft.com/api/stac/v1/search"
SOURCE_API = "planetary_computer_stac"


def radius_to_bbox(lat, lon, radius_km):
    """Approximate a [min_lon, min_lat, max_lon, max_lat] box around a point."""
    lat_delta = radius_km / 111.32
    cos_lat = math.cos(math.radians(lat))
    lon_delta = 180.0 if abs(cos_lat) < 1e-8 else radius_km / (111.32 * cos_lat)
    return [lon - lon_delta, lat - lat_delta, lon + lon_delta, lat + lat_delta]


BBOX = radius_to_bbox(LATITUDE, LONGITUDE, RADIUS_KM)

# Explicit, stable schema so every collection lands in a consistently typed
# Delta table even when a particular property is missing for that collection.
BRONZE_SCHEMA = StructType([
    StructField("item_id", StringType(), True),
    StructField("collection", StringType(), True),
    StructField("datetime_utc", StringType(), True),
    StructField("bbox_minx", DoubleType(), True),
    StructField("bbox_miny", DoubleType(), True),
    StructField("bbox_maxx", DoubleType(), True),
    StructField("bbox_maxy", DoubleType(), True),
    StructField("asset_count", IntegerType(), True),
    StructField("asset_keys", StringType(), True),
    StructField("cloud_cover", DoubleType(), True),
    StructField("platform", StringType(), True),
    StructField("properties_json", StringType(), True),
    StructField("source_api", StringType(), True),
    StructField("query_lat", DoubleType(), True),
    StructField("query_lon", DoubleType(), True),
    StructField("query_radius_km", DoubleType(), True),
    StructField("query_datetime", StringType(), True),
    StructField("ingested_at_utc", TimestampType(), True),
])


def _num(v):
    """Best-effort float cast; returns None on failure."""
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def ingest_collection(collection, target_table, datetime_range=None,
                      max_items=12, query=None):
    """Query the Planetary Computer STAC API for one collection over the AOI
    bbox and write the item metadata to a bronze Delta table (overwrite).

    Parameters
    ----------
    collection : str       STAC collection id (e.g. 'sentinel-1-rtc').
    target_table : str     Destination bronze Delta table name.
    datetime_range : str   Optional STAC datetime filter 'start/end'.
    max_items : int        Maximum number of items to request.
    query : dict           Optional STAC `query` extension filter.
    """
    payload = {"collections": [collection], "bbox": BBOX, "limit": max_items}
    if datetime_range:
        payload["datetime"] = datetime_range
    if query:
        payload["query"] = query

    resp = requests.post(STAC_SEARCH_URL, json=payload, timeout=90)
    resp.raise_for_status()
    features = resp.json().get("features", [])

    # Some collections ignore a datetime filter; if a filtered search returns
    # nothing, retry once with the bbox only so the table still lands.
    if not features and (datetime_range or query):
        retry = {"collections": [collection], "bbox": BBOX, "limit": max_items}
        resp = requests.post(STAC_SEARCH_URL, json=retry, timeout=90)
        resp.raise_for_status()
        features = resp.json().get("features", [])

    now = datetime.now(timezone.utc)
    rows = []
    for f in features:
        props = f.get("properties", {}) or {}
        b = f.get("bbox", [None, None, None, None]) or [None, None, None, None]
        assets = f.get("assets", {}) or {}
        rows.append((
            str(f.get("id")) if f.get("id") is not None else None,
            str(f.get("collection")) if f.get("collection") is not None else collection,
            str(props.get("datetime")) if props.get("datetime") is not None else None,
            _num(b[0]) if len(b) > 0 else None,
            _num(b[1]) if len(b) > 1 else None,
            _num(b[2]) if len(b) > 2 else None,
            _num(b[3]) if len(b) > 3 else None,
            int(len(assets)),
            ",".join(sorted(assets.keys())) if assets else None,
            _num(props.get("eo:cloud_cover")),
            str(props.get("platform")) if props.get("platform") is not None else None,
            json.dumps(props, default=str),
            SOURCE_API,
            float(LATITUDE),
            float(LONGITUDE),
            float(RADIUS_KM),
            datetime_range,
            now,
        ))

    df = spark.createDataFrame(rows, schema=BRONZE_SCHEMA)
    (df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(target_table))
    n = df.count()
    print(f"[{collection}] -> {target_table}: wrote {n} rows")
    return n


print("AOI:", AOI_NAME)
print("BBOX (min_lon, min_lat, max_lon, max_lat):", [round(c, 4) for c in BBOX])
print("Helpers ready: radius_to_bbox(), ingest_collection(), BRONZE_SCHEMA")

## 2. `sentinel-1-rtc` - Sentinel-1 Radiometrically Terrain Corrected
C-band radar backscatter corrected for terrain, so values are comparable
across slopes. Filtered to the 2024 summer window for alignment with the
Sentinel-2 optical scenes in the companion notebooks.

In [ ]:
ingest_collection(
    "sentinel-1-rtc",
    "bronze_sentinel_1_rtc",
    datetime_range="2024-06-01/2024-09-30",
    max_items=20,
)

## 3. `sentinel-1-grd` - Sentinel-1 Ground Range Detected
Detected VV/VH amplitude prior to radiometric terrain correction. Kept as a
raw radar reference alongside the RTC product. Same 2024 summer window.

In [ ]:
ingest_collection(
    "sentinel-1-grd",
    "bronze_sentinel_1_grd",
    datetime_range="2024-06-01/2024-09-30",
    max_items=20,
)

## 4. Verification
Confirm both Sentinel-1 bronze tables exist, report row counts, and peek at a
few records. Sentinel-1 polarizations and orbit properties are preserved in
the `properties_json` column.

In [ ]:
tables = ["bronze_sentinel_1_rtc", "bronze_sentinel_1_grd"]
for t in tables:
    try:
        c = spark.table(t).count()
        print(f"{t:26s} {c:6d} rows")
    except Exception as e:
        print(f"{t:26s} MISSING ({e.__class__.__name__})")

# Peek at a few Sentinel-1 records (polarizations live in properties_json)
spark.sql(
    "SELECT collection, item_id, datetime_utc, platform, asset_count "
    "FROM bronze_sentinel_1_rtc ORDER BY datetime_utc DESC LIMIT 10"
).show(truncate=False)